In [1]:
# ============================================================
# Finite Volume Method (FVM) Simulation in Python
# 2D Steady-State Heat Diffusion using Finite Volumes
# Visualization with PyVista
#
# Author: OpenAI ChatGPT
# ============================================================

# ------------------------------------------------------------
# INSTALLATION (uncomment if needed)
# ------------------------------------------------------------
# !pip install pyvista pyvistaqt

# ------------------------------------------------------------
# IMPORT LIBRARIES
# ------------------------------------------------------------
import numpy as np
import pyvista as pv

# ------------------------------------------------------------
# DOMAIN PARAMETERS
# ------------------------------------------------------------
Lx = 1.0          # Domain length in x
Ly = 1.0          # Domain length in y

Nx = 60           # Number of control volumes in x
Ny = 60           # Number of control volumes in y

dx = Lx / Nx
dy = Ly / Ny

# ------------------------------------------------------------
# PHYSICAL PARAMETERS
# ------------------------------------------------------------
k = 1.0           # Thermal conductivity

# Boundary conditions
T_left   = 100.0
T_right  = 0.0
T_top    = 50.0
T_bottom = 75.0

# ------------------------------------------------------------
# INITIALIZE TEMPERATURE FIELD
# ------------------------------------------------------------
T = np.zeros((Ny, Nx))

# Initial guess
T[:, :] = 50.0

# Apply Dirichlet boundary conditions
T[:, 0]    = T_left
T[:, -1]   = T_right
T[0, :]    = T_bottom
T[-1, :]   = T_top

# ------------------------------------------------------------
# FINITE VOLUME SOLVER
# ------------------------------------------------------------
max_iter = 5000
tolerance = 1e-6

print("Starting FVM iterations...")

for iteration in range(max_iter):

    T_old = T.copy()

    # Loop over internal control volumes
    for j in range(1, Ny - 1):
        for i in range(1, Nx - 1):

            # Neighbor temperatures
            Te = T[j, i + 1]
            Tw = T[j, i - 1]
            Tn = T[j + 1, i]
            Ts = T[j - 1, i]

            # Diffusion coefficients
            ae = k * dy / dx
            aw = k * dy / dx
            an = k * dx / dy
            as_ = k * dx / dy

            ap = ae + aw + an + as_

            # Finite Volume discretization
            T[j, i] = (ae * Te + aw * Tw + an * Tn + as_ * Ts) / ap

    # Reapply boundary conditions
    T[:, 0]    = T_left
    T[:, -1]   = T_right
    T[0, :]    = T_bottom
    T[-1, :]   = T_top

    # Convergence check
    error = np.max(np.abs(T - T_old))

    if iteration % 100 == 0:
        print(f"Iteration {iteration:5d} | Error = {error:.6e}")

    if error < tolerance:
        print("\nConverged!")
        print(f"Iterations: {iteration}")
        print(f"Final error: {error:.6e}")
        break

# ------------------------------------------------------------
# CREATE STRUCTURED GRID FOR PYVISTA
# ------------------------------------------------------------
x = np.linspace(0, Lx, Nx)
y = np.linspace(0, Ly, Ny)

X, Y = np.meshgrid(x, y)

Z = np.zeros_like(X)

grid = pv.StructuredGrid(X, Y, Z)

# PyVista expects flattened scalar arrays
grid["Temperature"] = T.flatten(order="F")

# ------------------------------------------------------------
# VISUALIZATION
# ------------------------------------------------------------
plotter = pv.Plotter(window_size=[1000, 700])

# Add scalar field
plotter.add_mesh(
    grid,
    scalars="Temperature",
    cmap="inferno",
    show_edges=False,
    smooth_shading=True,
)

# Add contour lines
contours = grid.contour(
    isosurfaces=15,
    scalars="Temperature"
)

plotter.add_mesh(
    contours,
    color="white",
    line_width=2,
)

# Add title
plotter.add_text(
    "2D Finite Volume Heat Diffusion",
    font_size=16,
)

# Camera view
plotter.view_xy()

# Add axes
plotter.add_axes()

# Show grid
plotter.show_grid()

# Render
plotter.show()

# ------------------------------------------------------------
# OPTIONAL: SAVE SCREENSHOT
# ------------------------------------------------------------
# plotter.screenshot("fvm_heat_diffusion.png")

print("\nSimulation complete.")

ModuleNotFoundError: No module named 'pyvista'